In [1]:
# =======================================================================
# SCRIPT 3: SUMARIZAÇÃO HIERÁRQUICA — VARIANTE MEDIASUM FT
# MediaSum LoRA para condensação + BitNet base para resposta final
# =======================================================================

# =======================================================================
# CÉLULA 0 — Configurações de memória
# =======================================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_SKIP_ALLOCATOR_WARMUP"] = "1"

In [2]:
# =======================================================================
# CÉLULA 1 — Instalação de dependências
# =======================================================================
!pip uninstall torchvision -y
!pip install -q -U transformers datasets peft accelerate
!pip install -q nltk rouge-score
!pip install -q "torchao>=0.16.0"

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 152.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.2 MB/s eta 0:00:00


In [3]:
# =======================================================================
# CÉLULA 2 — Imports
# =======================================================================
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import torch
import pandas as pd
import numpy as np
import json
import nltk
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList
from peft import PeftModel
from tqdm import tqdm
from google.colab import files as colab_files
import zipfile

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [4]:
# =======================================================================
# CÉLULA 3 — Configurações gerais
# =======================================================================
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T-bf16"
VARIANTE = "mediasum"
LORA_PATH = "./modelo_final_mediasum_v2"
OUTPUT_DIR = f"./resultados_hierarquico_{VARIANTE}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_TOKENS_CHUNK = 1024
MAX_TOKENS_CONDENSADO = 4096
BACKUP_A_CADA = 20

RETOMAR_DE_BACKUP = False
CAMINHO_BACKUP_RETOMADA = f"{OUTPUT_DIR}/resultados_{VARIANTE}.csv"
CAMINHO_CSV = os.path.join(OUTPUT_DIR, f"resultados_{VARIANTE}.csv")

SEED = 42

In [5]:
# =======================================================================
# CÉLULA 4 — Upload dos pesos LoRA do MediaSum
# =======================================================================
print("Faça upload do backup_mediasum_v2_completo.zip (pesos LoRA)")
uploaded = colab_files.upload()
with zipfile.ZipFile("backup_mediasum_v2_completo.zip", "r") as zip_ref:
    zip_ref.extractall("./")
print(f"Pesos LoRA disponíveis em: {LORA_PATH}")

Faça upload do backup_mediasum_v2_completo.zip (pesos LoRA)


Saving backup_mediasum_v2_completo.zip to backup_mediasum_v2_completo.zip
Pesos LoRA disponíveis em: ./modelo_final_mediasum_v2


In [6]:
# =======================================================================
# CÉLULA 5 — Download e carregamento do QMSum
# =======================================================================
from huggingface_hub import hf_hub_download
import zipfile as zf

print("Baixando QMSum...")
zip_path = hf_hub_download(
    repo_id="tau/scrolls",
    filename="qmsum.zip",
    repo_type="dataset"
)
os.makedirs("./qmsum_data", exist_ok=True)
with zf.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("./qmsum_data")

dados_test = []
with open("./qmsum_data/qmsum/test.jsonl", "r") as f:
    for linha in f:
        item = json.loads(linha)
        if not item.get("input") or not item.get("output"):
            continue
        dados_test.append({
            "input": item["input"],
            "output": item["output"]
        })

print(f"-> {len(dados_test)} instâncias carregadas")

Baixando QMSum...


qmsum.zip:   0%|          | 0.00/27.3M [00:00<?, ?B/s]

-> 281 instâncias carregadas


In [7]:
# =======================================================================
# CÉLULA 6 — Funções auxiliares
# =======================================================================

class StopOnToken(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids

    def __call__(self, input_ids, scores, **kwargs):
        for stop_id in self.stop_token_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

def get_stopping_criteria(tokenizer):
    stop_sequences = ["###", "##", "**", "Note:", "---"]
    stop_ids = []
    for seq in stop_sequences:
        ids = tokenizer.encode(seq, add_special_tokens=False)
        if ids:
            stop_ids.append(ids[0])
    return StoppingCriteriaList([StopOnToken(stop_ids)])

def limpar_output(texto):
    for seq in ["###", "##", "**", "Note:", "---"]:
        if seq in texto:
            texto = texto[:texto.index(seq)].strip()
    return texto

def gerar_texto(model, tokenizer, prompt, max_new_tokens,
                repetition_penalty, stopping_criteria):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096 - max_new_tokens
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=repetition_penalty,
            stopping_criteria=stopping_criteria
        )

    gerado = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return limpar_output(gerado)

def dividir_em_chunks(texto, tokenizer, max_tokens=1024):
    sentencas = nltk.sent_tokenize(texto)
    chunks = []
    chunk_atual = []
    tokens_atuais = 0

    for sentenca in sentencas:
        tokens_sentenca = len(tokenizer.encode(
            sentenca, add_special_tokens=False
        ))

        if tokens_sentenca > max_tokens:
            if chunk_atual:
                chunks.append(" ".join(chunk_atual))
                chunk_atual = []
                tokens_atuais = 0
            chunks.append(sentenca)
            continue

        if tokens_atuais + tokens_sentenca > max_tokens:
            if chunk_atual:
                chunks.append(" ".join(chunk_atual))
            chunk_atual = [sentenca]
            tokens_atuais = tokens_sentenca
        else:
            chunk_atual.append(sentenca)
            tokens_atuais += tokens_sentenca

    if chunk_atual:
        chunks.append(" ".join(chunk_atual))

    return chunks

def resumir_chunk(model, tokenizer, chunk, stopping_criteria):
    """Resume um chunk usando o MediaSum FT — treinado para gerar títulos jornalísticos."""
    prompt = (
        "### Instruction:\nWrite a short news headline summarizing the main "
        "topic of this excerpt in 16 words or less.\n\n"
        f"### Interview Transcript:\n{chunk}\n\n"
        "### Headline:\n"
    )
    return gerar_texto(
        model, tokenizer, prompt,
        max_new_tokens=50,       # Títulos curtos — menos tokens necessários
        repetition_penalty=1.3,
        stopping_criteria=stopping_criteria
    )

def responder_query(model, tokenizer, query,
                    documento_condensado, stopping_criteria):
    """Responde a query com o BitNet base."""
    prompt = (
        "### Instruction:\nWrite a concise summary answering the question "
        "based on the condensed meeting notes.\n\n"
        f"### Question:\n{query}\n\n"
        f"### Condensed Meeting Notes:\n{documento_condensado}\n\n"
        "### Answer:\n"
    )
    return gerar_texto(
        model, tokenizer, prompt,
        max_new_tokens=256,
        repetition_penalty=1.3,
        stopping_criteria=stopping_criteria
    )

def extrair_query_e_transcricao(input_texto):
    linhas = input_texto.split("\n", 1)
    query = linhas[0].strip()
    transcricao = linhas[1].strip() if len(linhas) > 1 else input_texto
    return query, transcricao

def fazer_backup(output_dir, variante):
    pasta_backup = f"./backup_temp_{variante}"
    os.makedirs(pasta_backup, exist_ok=True)
    shutil.copytree(output_dir, f"{pasta_backup}/resultados",
                    dirs_exist_ok=True)
    zip_nome = f"./backup_hierarquico_{variante}.zip"
    shutil.make_archive(zip_nome.replace(".zip", ""), "zip", pasta_backup)
    print(f"\nBackup salvo: {zip_nome} "
          f"({os.path.getsize(zip_nome) / (1024**2):.1f} MB)")
    colab_files.download(zip_nome)

In [8]:
# =======================================================================
# CÉLULA 7 — Carregamento dos modelos
# =======================================================================
print("Carregando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
stopping_criteria = get_stopping_criteria(tokenizer)

print("Carregando MediaSum FT para condensação...")
model_condensacao = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa"
)
model_condensacao = PeftModel.from_pretrained(model_condensacao, LORA_PATH)
model_condensacao.eval()
print("MediaSum FT carregado!")

print("\nCarregando BitNet base para etapa final...")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa"
)
model_base.eval()
print("BitNet base carregado!")

Carregando tokenizador...


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

MediaSum FT carregado!

Carregando BitNet base para etapa final...


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

BitNet base carregado!


In [9]:
# =======================================================================
# CÉLULA 7B — Retomada de backup (só rodar se RETOMAR_DE_BACKUP = True)
# =======================================================================
if RETOMAR_DE_BACKUP:
    print("Faça upload do backup para retomada:")
    uploaded = colab_files.upload()
    zip_backup = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_backup, "r") as zip_ref:
        zip_ref.extractall("./")

    # Verifica o caminho real do CSV
    for root, dirs, files in os.walk("./"):
        for f in files:
            if f == f"resultados_{VARIANTE}.csv":
                CAMINHO_BACKUP_RETOMADA = os.path.join(root, f)
                print(f"CSV encontrado em: {CAMINHO_BACKUP_RETOMADA}")

    df_progresso = pd.read_csv(CAMINHO_BACKUP_RETOMADA)
    indices_processados = set(df_progresso["instancia_idx"].tolist())
    resultados = df_progresso.to_dict("records")
    print(f"-> {len(resultados)} instâncias já processadas.")
    print(f"-> Retomando da instância {max(indices_processados)+1}...")
else:
    print("Iniciando do zero...")
    resultados = []
    indices_processados = set()

Iniciando do zero...


In [10]:
# =======================================================================
# CÉLULA 8 — Pipeline principal
# =======================================================================
print(f"\nIniciando pipeline hierárquico — variante: {VARIANTE.upper()}")
print(f"Total de instâncias: {len(dados_test)}")
print(f"Condensação: MediaSum FT | Etapa final: BitNet base")
print(f"Backup automático a cada {BACKUP_A_CADA} instâncias\n")

for i, item in enumerate(tqdm(dados_test, desc=f"Hierárquico ({VARIANTE})")):

    if i in indices_processados:
        continue

    query, transcricao = extrair_query_e_transcricao(item["input"])

    # 1. Divide em chunks respeitando sentenças
    chunks = dividir_em_chunks(transcricao, tokenizer, MAX_TOKENS_CHUNK)

    # 2. Resume cada chunk com MediaSum FT
    resumos_chunks = []
    for chunk in chunks:
        resumo_chunk = resumir_chunk(
            model_condensacao, tokenizer, chunk, stopping_criteria
        )
        resumos_chunks.append(resumo_chunk)

    # 3. Concatena os resumos intermediários
    documento_condensado = "\n".join(resumos_chunks)

    # Trunca por sentenças se necessário
    tokens_condensado = len(tokenizer.encode(
        documento_condensado, add_special_tokens=False
    ))
    if tokens_condensado > MAX_TOKENS_CONDENSADO:
        sentencas = nltk.sent_tokenize(documento_condensado)
        doc_truncado = []
        tokens_acc = 0
        for s in sentencas:
            t = len(tokenizer.encode(s, add_special_tokens=False))
            if tokens_acc + t > MAX_TOKENS_CONDENSADO:
                break
            doc_truncado.append(s)
            tokens_acc += t
        documento_condensado = " ".join(doc_truncado)
        tokens_condensado = tokens_acc

    # 4. Responde a query com BitNet base
    resumo_final = responder_query(
        model_base, tokenizer, query,
        documento_condensado, stopping_criteria
    )

    resultados.append({
        "instancia_idx": i,
        "query": query,
        "resumo_referencia": item["output"],
        "n_chunks": len(chunks),
        "tokens_condensado": tokens_condensado,
        "documento_condensado": documento_condensado,
        "resumos_chunks": " ||| ".join(resumos_chunks),
        "resumo_final": resumo_final
    })

    # Salva CSV progressivamente
    pd.DataFrame(resultados).to_csv(CAMINHO_CSV, index=False)

    # Backup automático a cada N instâncias
    processados_nesta_sessao = len(resultados) - len(indices_processados)
    if processados_nesta_sessao % BACKUP_A_CADA == 0:
        print(f"\n[Checkpoint] {len(resultados)} instâncias processadas")
        fazer_backup(OUTPUT_DIR, VARIANTE)

print(f"\nPipeline concluído! Total: {len(resultados)} instâncias")


Iniciando pipeline hierárquico — variante: MEDIASUM
Total de instâncias: 281
Condensação: MediaSum FT | Etapa final: BitNet base
Backup automático a cada 20 instâncias



Hierárquico (mediasum):   0%|          | 0/281 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


[Checkpoint] 20 instâncias processadas

Backup salvo: ./backup_hierarquico_mediasum.zip (0.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...


[Checkpoint] 280 instâncias processadas

Backup salvo: ./backup_hierarquico_mediasum.zip (0.2 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Hierárquico (mediasum): 100%|█████████▉| 280/281 [8:25:30<00:57, 57.95s/it][transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. P


Pipeline concluído! Total: 281 instâncias


In [11]:
# =======================================================================
# CÉLULA 9 — Verificação de exemplos
# =======================================================================
df = pd.read_csv(CAMINHO_CSV)
print(f"Total salvo: {len(df)} instâncias\n")

for i in range(min(3, len(df))):
    print(f"\n{'='*60}")
    print(f"INSTÂNCIA {i+1}")
    print(f"Chunks: {df.iloc[i]['n_chunks']} | "
          f"Tokens condensado: {df.iloc[i]['tokens_condensado']}")
    print(f"\nQUERY:\n{df.iloc[i]['query']}")
    print(f"\nREFERÊNCIA:\n{df.iloc[i]['resumo_referencia']}")
    print(f"\nRESUMO FINAL:\n{df.iloc[i]['resumo_final']}")

Total salvo: 281 instâncias


INSTÂNCIA 1
Chunks: 13 | Tokens condensado: 484

QUERY:
Summarize the discussion about the efficacy of the law.

REFERÊNCIA:
Barry Hughes first stated that children had fewer rights than adults and therefore the law should be enforced to defend physical assault. As such social behavior was not available now, the law should change to reflect that. The discussion then turned to talk about the legal framework and its prosecution.

RESUMO FINAL:
During the discussions, various topics were addressed:

1.

INSTÂNCIA 2
Chunks: 13 | Tokens condensado: 484

QUERY:
What did Barry Hughes think about the legal framework when talking about the efficacy of the law?

REFERÊNCIA:
Barry thought that the legal framework would make things clearer for parents and professionals. But when it came to prosecuting, there was a degree of confusion and some cases were in the grey areas.

RESUMO FINAL:
Based on the provided information, it is not explicitly stated what specific thoug

In [12]:
# =======================================================================
# CÉLULA 10 — Backup final completo
# =======================================================================
fazer_backup(OUTPUT_DIR, VARIANTE)
print("Script concluído!")


Backup salvo: ./backup_hierarquico_mediasum.zip (0.2 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Script concluído!
